# PROJECT FORESIGHT — NOTEBOOK 07
## Deployed Scoring Service (D6)

### Objective

This notebook prepares and validates the prediction service for Project FORESIGHT.

The service exposes:

1. Forecast prediction API
2. Inventory risk scoring API
3. Health-check endpoint

### Acceptance Criteria

- Hosted prediction service
- Forecast API
- Risk API
- Public URL
- Smoke test

### Inputs

- Random Forest forecasting model from Notebook 04
- Risk-scoring logic from Notebook 05
- Dashboard-ready outputs from Notebook 06

### Outputs

- FastAPI application
- Forecast endpoint
- Risk endpoint
- Health endpoint
- Local API validation
- Deployment-ready service

## 1. Service Architecture

The deployed scoring service follows this flow:

Client
↓
FastAPI
↓
Forecast Model
↓
Forecast Result
↓
Risk Scoring Logic
↓
Business Decision

The service is intentionally kept separate from the Streamlit dashboard.

The dashboard is responsible for visualization and planning.

The API is responsible for prediction and scoring.

In [1]:
import sys
import pandas as pd
import joblib
import fastapi
import uvicorn

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("FastAPI:", fastapi.__version__)
print("Joblib:", joblib.__version__)

Python: 3.14.6 (tags/v3.14.6:c63aec6, Jun 10 2026, 10:26:10) [MSC v.1944 64 bit (AMD64)]
Pandas: 3.0.5
FastAPI: 0.141.1
Joblib: 1.5.3


## 2. Project Paths

The service uses the existing FORESIGHT project structure.

The forecasting model is loaded from the models directory.

Risk scoring remains transparent and rule-based, following the decision logic established in Notebook 05.

In [4]:
from pathlib import Path

ROOT_DIR = Path.cwd().parent

MODEL_PATH = (
    ROOT_DIR
    / "data"
    / "processed"
    / "models"
    / "random_forest_forecaster.pkl"
)

PROCESSED_DIR = ROOT_DIR / "data" / "processed"

print("Root:", ROOT_DIR)
print("Model:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

Root: c:\Users\Dell\OneDrive\Project_Foresight
Model: c:\Users\Dell\OneDrive\Project_Foresight\data\processed\models\random_forest_forecaster.pkl
Model exists: True


In [3]:
from pathlib import Path

ROOT_DIR = Path.cwd().parent

matches = list(
    ROOT_DIR.rglob("random_forest_forecaster.pkl")
)

print("Models found:")

for path in matches:
    print(path)

Models found:
c:\Users\Dell\OneDrive\Project_Foresight\data\processed\models\random_forest_forecaster.pkl


In [5]:
import joblib

model = joblib.load(MODEL_PATH)

print("Model loaded successfully!")
print("Model type:", type(model))

Model loaded successfully!
Model type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [6]:
if hasattr(model, "n_features_in_"):
    print("Expected features:", model.n_features_in_)

if hasattr(model, "feature_names_in_"):
    print("\nModel features:")
    print(list(model.feature_names_in_))

Expected features: 10

Model features:
['sell_price', 'has_event', 'has_snap', 'is_weekend', 'lag_1', 'lag_2', 'lag_4', 'lag_8', 'rolling_mean_4', 'rolling_std_4']


In [7]:
FEATURES = [
    "sell_price",
    "has_event",
    "has_snap",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "rolling_mean_4",
    "rolling_std_4"
]

print("Number of API features:", len(FEATURES))
print("Model expects:", model.n_features_in_)

assert len(FEATURES) == model.n_features_in_

print("\nFeature order:")
for i, feature in enumerate(FEATURES, start=1):
    print(f"{i}. {feature}")

print("\nFeature validation passed! ✅")

Number of API features: 10
Model expects: 10

Feature order:
1. sell_price
2. has_event
3. has_snap
4. is_weekend
5. lag_1
6. lag_2
7. lag_4
8. lag_8
9. rolling_mean_4
10. rolling_std_4

Feature validation passed! ✅


In [8]:
import pandas as pd

test_input = pd.DataFrame([{
    "sell_price": 2.50,
    "has_event": 0,
    "has_snap": 1,
    "is_weekend": 0,
    "lag_1": 20,
    "lag_2": 22,
    "lag_4": 19,
    "lag_8": 21,
    "rolling_mean_4": 20.5,
    "rolling_std_4": 2.1
}])

test_input = test_input[FEATURES]

prediction = model.predict(test_input)

print("Forecast prediction:", float(prediction[0]))

Forecast prediction: 7.661256557625507


In [9]:
def forecast_prediction(input_data):
    
    df = pd.DataFrame([input_data])
    
    # Guarantee correct feature order
    df = df[FEATURES]
    
    prediction = model.predict(df)
    
    return float(prediction[0])


result = forecast_prediction({
    "sell_price": 2.50,
    "has_event": 0,
    "has_snap": 1,
    "is_weekend": 0,
    "lag_1": 20,
    "lag_2": 22,
    "lag_4": 19,
    "lag_8": 21,
    "rolling_mean_4": 20.5,
    "rolling_std_4": 2.1
})

print("Forecast:", result)

Forecast: 7.661256557625507


## 5. Forecast API Contract

The `/predict/forecast` endpoint accepts the ten features required by the Random Forest model.

Request:

{
    "sell_price": 2.50,
    "has_event": 0,
    "has_snap": 1,
    "is_weekend": 0,
    "lag_1": 20,
    "lag_2": 22,
    "lag_4": 19,
    "lag_8": 21,
    "rolling_mean_4": 20.5,
    "rolling_std_4": 2.1
}

Response:

{
    "forecast": 21.37
}

The API preserves the feature ordering used during model training.

In [11]:
from pydantic import BaseModel

In [12]:
class ForecastRequest(BaseModel):
    features: list[float]

In [13]:
from pydantic import BaseModel


class ForecastRequest(BaseModel):

    sell_price: float
    has_event: int
    has_snap: int
    is_weekend: int
    lag_1: float
    lag_2: float
    lag_4: float
    lag_8: float
    rolling_mean_4: float
    rolling_std_4: float


class RiskRequest(BaseModel):

    estimated_inventory: float
    safety_stock: float
    forecast: float


print("Request models created successfully! ✅")

Request models created successfully! ✅


In [14]:
import pandas as pd
import joblib

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

print("All API imports loaded successfully! ✅")

All API imports loaded successfully! ✅


In [15]:
test_input = pd.DataFrame([{
    "sell_price": 2.50,
    "has_event": 0,
    "has_snap": 1,
    "is_weekend": 0,
    "lag_1": 20,
    "lag_2": 22,
    "lag_4": 19,
    "lag_8": 21,
    "rolling_mean_4": 20.5,
    "rolling_std_4": 2.1
}])

test_input = test_input[FEATURES]

prediction = model.predict(test_input)

print("Forecast prediction:", float(prediction[0]))

Forecast prediction: 7.6612565576255065


In [16]:
from pathlib import Path

api_path = ROOT_DIR / "api.py"

api_code = r'''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pathlib import Path
import joblib
import pandas as pd


# ============================================================
# PROJECT PATHS
# ============================================================

ROOT_DIR = Path(__file__).resolve().parent

MODEL_PATH = (
    ROOT_DIR
    / "data"
    / "processed"
    / "models"
    / "random_forest_forecaster.pkl"
)


# ============================================================
# LOAD MODEL
# ============================================================

model = joblib.load(MODEL_PATH)


FEATURES = [
    "sell_price",
    "has_event",
    "has_snap",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "rolling_mean_4",
    "rolling_std_4"
]


# ============================================================
# FASTAPI APPLICATION
# ============================================================

app = FastAPI(
    title="Project FORESIGHT Scoring Service",
    description="Forecast and inventory risk prediction API",
    version="1.0.0"
)


# ============================================================
# REQUEST SCHEMAS
# ============================================================

class ForecastRequest(BaseModel):

    sell_price: float
    has_event: int
    has_snap: int
    is_weekend: int
    lag_1: float
    lag_2: float
    lag_4: float
    lag_8: float
    rolling_mean_4: float
    rolling_std_4: float


class RiskRequest(BaseModel):

    estimated_inventory: float
    safety_stock: float
    forecast: float


# ============================================================
# HEALTH CHECK
# ============================================================

@app.get("/health")
def health():

    return {
        "status": "healthy",
        "service": "Project FORESIGHT Scoring Service",
        "model_loaded": True
    }


# ============================================================
# FORECAST API
# ============================================================

@app.post("/predict/forecast")
def predict_forecast(request: ForecastRequest):

    try:

        input_data = pd.DataFrame([{
            "sell_price": request.sell_price,
            "has_event": request.has_event,
            "has_snap": request.has_snap,
            "is_weekend": request.is_weekend,
            "lag_1": request.lag_1,
            "lag_2": request.lag_2,
            "lag_4": request.lag_4,
            "lag_8": request.lag_8,
            "rolling_mean_4": request.rolling_mean_4,
            "rolling_std_4": request.rolling_std_4
        }])

        input_data = input_data[FEATURES]

        prediction = model.predict(input_data)

        return {
            "forecast": float(prediction[0])
        }

    except Exception as e:

        raise HTTPException(
            status_code=400,
            detail=str(e)
        )


# ============================================================
# RISK API
# ============================================================

@app.post("/predict/risk")
def predict_risk(request: RiskRequest):

    inventory_gap = (
        request.estimated_inventory
        - request.safety_stock
    )

    if inventory_gap < 0:

        risk = "Stockout Risk"
        priority = "High"
        decision = "Reorder Immediately"
        recommended_action = "Increase replenishment"

    elif request.estimated_inventory > (
        request.forecast * 1.5
    ):

        risk = "Overstock Risk"
        priority = "Medium"
        decision = "Reduce Purchasing / Launch Promotion"
        recommended_action = (
            "Reduce purchasing or promote stock"
        )

    else:

        risk = "Healthy"
        priority = "Low"
        decision = "Maintain Current Inventory"
        recommended_action = (
            "Maintain current inventory"
        )

    return {

        "risk": risk,
        "priority": priority,
        "decision": decision,
        "recommended_action": recommended_action,
        "inventory_gap": inventory_gap

    }
'''

api_path.write_text(
    api_code,
    encoding="utf-8"
)

print("API created successfully!")
print(api_path)

API created successfully!
c:\Users\Dell\OneDrive\Project_Foresight\api.py


In [17]:
print("API exists:", api_path.exists())

API exists: True


## 6. Local API Deployment

The FastAPI application is now ready to run locally.

The local service will expose:

- `/health`
- `/predict/forecast`
- `/predict/risk`

Interactive Swagger documentation will be available at `/docs`.

In [18]:
model_size_mb = MODEL_PATH.stat().st_size / (1024 ** 2)

print(f"Model size: {model_size_mb:.2f} MB")

Model size: 177.23 MB
